# Reinforcement Learning with Function Approximation

This notebook shows a small reinforcement learning example where the agent learns to reach a goal in a 2D GridWorld.

Instead of using a Q-table, we approximate the action-value function with a simple linear function approximator:

\[
Q(s, a) \approx w_a^T \phi(s)
\]

where \(\phi(s)\) is a vector of radial basis function features built from the 2D state coordinates.

The notebook includes:

- a small 2D environment with obstacles,
- approximate Q-learning,
- a learned value heatmap,
- policy arrows,
- an animation of learning progress.

## Libraries

The notebook uses only common Python libraries:

```python
numpy
matplotlib
IPython
```

No external API is required.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

## 1. Define a simple 2D GridWorld

The agent starts at `S`, must reach `G`, and cannot move through obstacles.

Rewards:

- `+1.0` when reaching the goal
- `-0.02` for a normal step
- `-0.20` for hitting a wall or obstacle

In [ ]:
class GridWorld:
    def __init__(self):
        self.rows = 5
        self.cols = 7
        self.start = (0, 0)
        self.goal = (4, 6)
        self.obstacles = {(1, 2), (2, 2), (3, 2), (3, 4)}

        # Actions: 0=up, 1=down, 2=left, 3=right
        self.actions = {
            0: (-1, 0),
            1: (1, 0),
            2: (0, -1),
            3: (0, 1),
        }

        self.action_names = {
            0: "up",
            1: "down",
            2: "left",
            3: "right",
        }

        self.reset()

    def reset(self):
        self.agent = self.start
        return self.agent

    def normalized(self, state):
        """Convert grid coordinates into normalized coordinates in [0, 1]."""
        r, c = state
        return np.array([
            r / (self.rows - 1),
            c / (self.cols - 1)
        ], dtype=float)

    def is_valid(self, state):
        r, c = state
        inside_grid = 0 <= r < self.rows and 0 <= c < self.cols
        return inside_grid and state not in self.obstacles

    def step(self, action):
        dr, dc = self.actions[action]
        r, c = self.agent
        candidate = (r + dr, c + dc)

        reward = -0.02
        done = False

        # If the move is invalid, stay in the same position and receive a penalty
        if not self.is_valid(candidate):
            candidate = self.agent
            reward = -0.20

        # Goal reached
        if candidate == self.goal:
            reward = 1.0
            done = True

        self.agent = candidate
        return candidate, reward, done


env = GridWorld()
print("Grid size:", env.rows, "x", env.cols)
print("Start:", env.start)
print("Goal:", env.goal)
print("Obstacles:", env.obstacles)

## 2. Define the function approximator

We approximate the Q-function using radial basis function features.

The state is a 2D coordinate:

\[
s = (row, col)
\]

The feature function maps it into a richer representation:

\[
\phi(s) = [1, \text{RBF}_1(s), \text{RBF}_2(s), ..., \text{RBF}_n(s)]
\]

Then each action has its own weights:

\[
Q(s,a) = w_a^T \phi(s)
\]

This is function approximation because the value is predicted by a function, not stored directly in a table for every `(state, action)` pair.

In [ ]:
def make_rbf_centers(rows=4, cols=5):
    """
    Create a small grid of RBF centers in normalized coordinate space.
    Fewer centers make the approximation smoother.
    """
    centers = []
    for r in np.linspace(0, 1, rows):
        for c in np.linspace(0, 1, cols):
            centers.append([r, c])
    return np.array(centers)


class RBFQApproximator:
    def __init__(self, num_actions, centers, sigma=0.22):
        self.num_actions = num_actions
        self.centers = centers
        self.sigma = sigma

        # +1 because we add a bias feature
        self.num_features = len(centers) + 1

        # One weight vector per action
        self.W = np.zeros((num_actions, self.num_features))

    def features(self, state_xy):
        """Compute RBF features for one normalized state."""
        diff = self.centers - state_xy[None, :]
        squared_distance = np.sum(diff * diff, axis=1)
        rbf = np.exp(-squared_distance / (2 * self.sigma ** 2))

        # Add bias feature at the beginning
        return np.concatenate([[1.0], rbf])

    def q_values(self, state_xy):
        """Return Q(s, a) for all actions."""
        phi = self.features(state_xy)
        return self.W @ phi

    def update(self, state_xy, action, target, alpha):
        """Semi-gradient update for one action."""
        phi = self.features(state_xy)
        prediction = self.W[action] @ phi
        td_error = target - prediction

        # Gradient step: w_a <- w_a + alpha * td_error * phi(s)
        self.W[action] += alpha * td_error * phi

        return td_error, prediction


centers = make_rbf_centers(rows=4, cols=5)
qfunc = RBFQApproximator(num_actions=4, centers=centers, sigma=0.22)

print("Number of RBF centers:", len(centers))
print("Number of features:", qfunc.num_features)

## 3. Helper functions for policy, rollout, and visualization

In [ ]:
def epsilon_greedy(approximator, env, state, epsilon):
    """
    Choose a random action with probability epsilon,
    otherwise choose the action with maximum predicted Q-value.
    """
    if random.random() < epsilon:
        return random.randrange(4)

    q_values = approximator.q_values(env.normalized(state))
    return int(np.argmax(q_values))


def greedy_rollout(env, approximator, max_steps=80):
    """Run the current greedy policy from the start state."""
    state = env.reset()
    path = [state]
    total_reward = 0.0

    for _ in range(max_steps):
        action = epsilon_greedy(approximator, env, state, epsilon=0.0)
        state, reward, done = env.step(action)
        path.append(state)
        total_reward += reward

        if done:
            break

    return path, total_reward


def compute_value_policy(env, approximator):
    """Compute max_a Q(s,a) and greedy action for each grid cell."""
    values = np.full((env.rows, env.cols), np.nan)
    policy = np.full((env.rows, env.cols), "", dtype=object)

    arrows = {
        0: "↑",
        1: "↓",
        2: "←",
        3: "→",
    }

    for r in range(env.rows):
        for c in range(env.cols):
            state = (r, c)

            if state in env.obstacles:
                policy[r, c] = "#"
                continue

            if state == env.goal:
                values[r, c] = 1.0
                policy[r, c] = "G"
                continue

            q_values = approximator.q_values(env.normalized(state))
            best_action = int(np.argmax(q_values))
            values[r, c] = np.max(q_values)
            policy[r, c] = arrows[best_action]

    return values, policy

## 4. Train with approximate Q-learning

The target is:

\[
\text{target} = r + \gamma \max_{a'} Q(s', a')
\]

The prediction is:

\[
Q(s,a)
\]

The temporal-difference error is:

\[
\delta = \text{target} - Q(s,a)
\]

Then the weights for the selected action are updated with:

\[
w_a \leftarrow w_a + \alpha \delta \phi(s)
\]

In [ ]:
# Recreate environment and approximator so this cell is safely rerunnable
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

env = GridWorld()
centers = make_rbf_centers(rows=4, cols=5)
qfunc = RBFQApproximator(num_actions=4, centers=centers, sigma=0.22)

alpha = 0.05
gamma = 0.95
epsilon = 1.00
epsilon_min = 0.05
epsilon_decay = 0.992

episodes = 800
max_steps = 80
snapshot_every = 50

reward_history = []
td_error_history = []
snapshots = []

for episode in range(episodes + 1):
    state = env.reset()
    total_reward = 0.0
    td_errors = []

    for step_idx in range(max_steps):
        action = epsilon_greedy(qfunc, env, state, epsilon)
        next_state, reward, done = env.step(action)

        target = reward
        if not done:
            next_q_values = qfunc.q_values(env.normalized(next_state))
            target += gamma * np.max(next_q_values)

        td_error, prediction = qfunc.update(
            state_xy=env.normalized(state),
            action=action,
            target=target,
            alpha=alpha,
        )

        td_errors.append(abs(td_error))
        total_reward += reward
        state = next_state

        if done:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    reward_history.append(total_reward)
    td_error_history.append(np.mean(td_errors) if td_errors else 0.0)

    if episode % snapshot_every == 0:
        values, policy = compute_value_policy(env, qfunc)
        path, greedy_reward = greedy_rollout(env, qfunc)

        snapshots.append({
            "episode": episode,
            "values": values.copy(),
            "policy": policy.copy(),
            "path": path,
            "greedy_reward": greedy_reward,
            "reward_history": reward_history.copy(),
            "td_error_history": td_error_history.copy(),
        })

print("Training complete.")
print("Final greedy path:", snapshots[-1]["path"])
print("Final greedy reward:", round(snapshots[-1]["greedy_reward"], 3))

## 5. Plot reward and TD error

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Training reward over time")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(td_error_history)
plt.xlabel("Episode")
plt.ylabel("Mean absolute TD error")
plt.title("TD error over time")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Final learned value heatmap and policy arrows

The heatmap shows:

\[
V(s) \approx \max_a Q(s,a)
\]

The arrows show the greedy policy induced by the learned Q-function.

In [ ]:
def plot_value_and_policy(env, values, policy, path=None, title="Learned value function and policy"):
    fig, ax = plt.subplots(figsize=(8, 5))

    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="black")

    im = ax.imshow(values, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Estimated value")

    # Draw grid lines
    ax.set_xticks(np.arange(-0.5, env.cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, env.rows, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Ticks as coordinates
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

    # Policy arrows and labels
    for r in range(env.rows):
        for c in range(env.cols):
            text = policy[r, c]
            if text == "#":
                ax.text(c, r, "#", ha="center", va="center", fontsize=18, color="white", fontweight="bold")
            elif text == "G":
                ax.text(c, r, "G", ha="center", va="center", fontsize=18, color="white", fontweight="bold")
            else:
                ax.text(c, r, text, ha="center", va="center", fontsize=18, color="white", fontweight="bold")

    # Optional path overlay
    if path is not None and len(path) > 1:
        ys = [p[0] for p in path]
        xs = [p[1] for p in path]
        ax.plot(xs, ys, marker="o", linewidth=3, color="red", alpha=0.85, label="Greedy path")
        ax.legend(loc="upper right")

    ax.set_title(title)
    plt.show()


final = snapshots[-1]
plot_value_and_policy(
    env,
    final["values"],
    final["policy"],
    path=final["path"],
    title=f"Final value heatmap, policy arrows, and greedy path - episode {final['episode']}"
)

## 7. Animate the learning progress

This animation shows how the learned value function and greedy policy evolve during training.

The left panel shows:

- value heatmap,
- policy arrows,
- the current greedy path.

The right panel shows the reward curve up to the current episode.

In [ ]:
def create_learning_animation(env, snapshots, interval=900):
    fig, (ax_grid, ax_reward) = plt.subplots(
        1, 2,
        figsize=(13, 5),
        gridspec_kw={"width_ratios": [1.0, 1.25]}
    )

    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="black")

    all_values = np.concatenate([
        snap["values"][~np.isnan(snap["values"])]
        for snap in snapshots
    ])
    vmin, vmax = np.min(all_values), np.max(all_values)

    def draw(frame_idx):
        snap = snapshots[frame_idx]
        values = snap["values"]
        policy = snap["policy"]
        path = snap["path"]
        episode = snap["episode"]
        rewards_so_far = snap["reward_history"]

        ax_grid.clear()
        ax_reward.clear()

        # Left: value heatmap and policy
        ax_grid.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax)

        ax_grid.set_xticks(np.arange(-0.5, env.cols, 1), minor=True)
        ax_grid.set_yticks(np.arange(-0.5, env.rows, 1), minor=True)
        ax_grid.grid(which="minor", color="white", linestyle="-", linewidth=1.3)
        ax_grid.tick_params(which="minor", bottom=False, left=False)
        ax_grid.set_xticks(range(env.cols))
        ax_grid.set_yticks(range(env.rows))
        ax_grid.set_xlabel("Column")
        ax_grid.set_ylabel("Row")
        ax_grid.set_title(f"Episode {episode}: value and greedy policy")

        for r in range(env.rows):
            for c in range(env.cols):
                text = policy[r, c]
                if text == "#":
                    ax_grid.text(c, r, "#", ha="center", va="center", fontsize=16, color="white", fontweight="bold")
                elif text == "G":
                    ax_grid.text(c, r, "G", ha="center", va="center", fontsize=16, color="white", fontweight="bold")
                else:
                    ax_grid.text(c, r, text, ha="center", va="center", fontsize=16, color="white", fontweight="bold")

        # Draw greedy path
        if path is not None and len(path) > 1:
            ys = [p[0] for p in path]
            xs = [p[1] for p in path]
            ax_grid.plot(xs, ys, marker="o", linewidth=3, color="red", alpha=0.85)

            # Emphasize start state
            ax_grid.scatter(xs[0], ys[0], s=180, marker="s", color="white", edgecolor="black", zorder=5)
            ax_grid.text(xs[0], ys[0], "S", ha="center", va="center", fontsize=11, color="black", fontweight="bold", zorder=6)

        # Right: reward curve so far
        ax_reward.plot(rewards_so_far, linewidth=2)
        ax_reward.set_xlim(0, snapshots[-1]["episode"])
        ax_reward.set_ylim(min(reward_history) - 0.2, max(reward_history) + 0.2)
        ax_reward.set_xlabel("Episode")
        ax_reward.set_ylabel("Training episode reward")
        ax_reward.set_title("Reward curve during learning")
        ax_reward.grid(True, alpha=0.3)

        ax_reward.axvline(episode, linestyle="--", alpha=0.6)
        ax_reward.text(
            0.02, 0.92,
            f"Greedy rollout reward: {snap['greedy_reward']:.2f}",
            transform=ax_reward.transAxes,
            fontsize=11,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8}
        )

        return []

    anim = FuncAnimation(
        fig,
        draw,
        frames=len(snapshots),
        interval=interval,
        blit=False,
        repeat=True
    )

    plt.close(fig)
    return anim


anim = create_learning_animation(env, snapshots, interval=900)
HTML(anim.to_jshtml())

## 8. Optional: export the animation as a GIF

This requires Pillow. If Pillow is missing, install it with:

```python
pip install pillow
```

In [ ]:
# Uncomment this cell if you want to save the animation as a GIF.
# from matplotlib.animation import PillowWriter
# anim.save("rl_function_approximation_learning.gif", writer=PillowWriter(fps=1))

## What to modify

Useful things to experiment with:

- `episodes`: more episodes usually improve the policy.
- `alpha`: learning rate.
- `gamma`: discount factor.
- `epsilon_decay`: controls how quickly exploration decreases.
- `sigma`: controls the smoothness of the RBF approximation.
- `env.obstacles`: change the map layout.
- `env.goal`: move the target state.

A lower `sigma` makes the function approximator more local. A higher `sigma` makes the learned values smoother across the grid.